# Biomarker Discovery 

> *Sequential multi-disorder analysis using permutation importance to identify transdiagnostic neuroimaging biomarkers across 414 brain regions.*

### Workflow
`Load Data` → `Train/Fine-tune BBTransformer` → `Validate (F1 ≥ 0.65)` → `Permutation Ranking (50×)` → `Export ROI Importance` → `Propagate Weights`


### Key Outputs
📄 `importance_<disorder>.csv` · Top-ranked ROIs per condition  
📈 `<disorder>_results.json` · Full performance metrics & confusion matrices  
⚖️ `weights_<disorder>.pth` · Transferable model weights for sequential learning  

> *All analyses reproducible with `random_seed=42`; importance scores reflect F1-drop upon region-wise shuffling.*

In [1]:
# bbtransformer_analyzer.py 

import os
from pathlib import Path
from typing import List, Optional, Dict, Any, Union
from bbtransformer import run_analysis



class BBTransformerAnalyzer:
    def __init__(
        self,
        base_dir: str,
        weights_dir: str = "weights",
        results_dir: str = "results",
        initial_weights: Optional[str] = None,
        min_composite: float = 0.65,
        max_trials_per_disorder: int = 50
    ):
        self.base_dir = Path(base_dir)
        self.weights_dir = Path(weights_dir)
        self.results_dir = Path(results_dir)
        self.weights_dir.mkdir(exist_ok=True)
        self.results_dir.mkdir(exist_ok=True)
        
        self.current_weights = initial_weights
        self.min_composite = min_composite
        self.max_trials = max_trials_per_disorder
        self.valid_models = []

    def get_chrt_paths(self, disorder: str):
        """Resolve CHRT-style paths."""
        return (
            self.base_dir / f"fmri_{disorder}.npz",
            self.base_dir / f"pheno_{disorder}.csv"
        )

    def resolve_paths(self, task_spec: Union[str, Dict[str, str]]):
        """
        Resolve data paths from either:
          - str: disorder name → use CHRT convention
          - dict: {'data_path': ..., 'pheno_path': ...}
        """
        if isinstance(task_spec, str):
            # Assume CHRT-style disorder name
            return self.get_chrt_paths(task_spec)
        elif isinstance(task_spec, dict):
            # Explicit paths
            if 'data_path' not in task_spec or 'pheno_path' not in task_spec:
                raise ValueError("Dict must contain 'data_path' and 'pheno_path'")
            return Path(task_spec['data_path']), Path(task_spec['pheno_path'])
        else:
            raise TypeError("task_spec must be str or dict")

    def is_valid(self, metrics: Dict[str, float]) -> bool:
        return all(
            metrics.get(metric, 0) >= self.min_composite
            for metric in ['f1', 'roc_auc', 'accuracy', 'precision', 'recall']
        )

    def run_ordered_pipeline(self, tasks: List[Union[str, Dict[str, str]]]) -> Dict[str, Any]:
        """
        Run pipeline over mixed task specs with robust weight propagation.
        """
        results_summary = {}

        for i, task in enumerate(tasks, 1):
            # Extract display name
            if isinstance(task, str):
                disorder_name = task
            else:
                disorder_name = task.get('name', 'unnamed_task')

            print(f"\n{'='*70}")
            print(f"PHASE {i}/{len(tasks)}: {disorder_name}")
            print(f"{'='*70}")

            data_path, pheno_path = self.resolve_paths(task)
            
            if not data_path.exists():
                print(f"  ❌ Data not found: {data_path}")
                continue
            if not pheno_path.exists():
                print(f"  ❌ Phenotype not found: {pheno_path}")
                continue

            # Determine if pretrained weights are available
            use_pretrained = self.current_weights is not None
            if use_pretrained:
                print(f"  📥 Loading pretrained weights: {self.current_weights}")
            else:
                print(f"  🆕 Initializing from scratch (No prior weights)")

            best_result = None

            for trial in range(self.max_trials):
                print(f"  Trial {trial+1}/{self.max_trials}...")

                try:
                    result = run_analysis(
                        target_column=disorder_name,
                        data_path=str(data_path),
                        pheno_path=str(pheno_path),
                        use_pretrained=use_pretrained,
                        pretrained_weight_file=self.current_weights,
                        compute_importance=True,
                        importance_n_repeats = 50,
                        importance_metric='f1',
                        random_seed=42 + trial,
                        weights_dir=str(self.weights_dir)
                    )
                    
                    if self.is_valid(result['metrics']):
                        best_result = result
                        print(f"  ✅ VALID MODEL FOUND (Composite: {result['metrics']['f1']:.4f})")
                        break
                    else:
                        print(f"  ❌ Trial {trial+1} failed validity check")
                        
                except Exception as e:
                    print(f"  ❌ Trial {trial+1} crashed: {str(e)}")
                    continue

            results_summary[disorder_name] = {
                'valid': best_result is not None,
                'metrics': best_result['metrics'] if best_result else None,
                'weights_used': self.current_weights,
                'weights_saved': None
            }

            if best_result is not None:
                # Update weights only if a valid model is found
                weight_file = f"weights_{disorder_name}.pth"
                new_weight_path = str(self.weights_dir / weight_file)
                
                # Verify the file exists before updating pointer
                if os.path.exists(new_weight_path):
                    self.current_weights = new_weight_path
                    results_summary[disorder_name]['weights_saved'] = self.current_weights
                    self.valid_models.append(disorder_name)
                    print(f"  🔁 Weights updated for next disorder: {self.current_weights}")
                else:
                    print(f"  ⚠️ Valid model found, but weight file not located at expected path.")
            else:
                # Task failed validity checks
                if self.current_weights is not None:
                    print(f"  ⚠️ Task failed validity. Retaining previous weights: {self.current_weights}")
                else:
                    print(f"  🧼 No prior weights available. Next disorder will train from scratch.")
                    # Do not explicitly set to None again as it is already None, 
                    # but ensure logic doesn't accidentally clear it if it was set.

        return results_summary

In [3]:

TASKS = [

    # External datasets (explicit paths)
    {
        'name': 'ASD',
        'data_path': '/mnt/movement/users/jaizor/xtra/data/fmri/abide/fmri_ASD.npz',
        'pheno_path': '/mnt/movement/users/jaizor/xtra/data/fmri/abide/pheno_ASD.csv'
    },

    # Neurodegenerative (focal, high signal)
    'NervousSystem_Dementia_Developmental',
    'Psychopathology_Dementia',
    'Psychopathology_Organic_Mental_Disorder',


    # Vascular & inflammatory (network disruption)
    'NervousSystem_Inflammatory_Infectious',
    'NervousSystem_Cerebrovascular',



    # Psychiatric (distributed, heterogeneous)
    'ICD_F31_Bipolar',
    'Psychopathology_Schizophrenia_Spectrum',


    # Epilepsy & paroxysmal (temporal instability)
    'NervousSystem_Epilepsy_Status_Epilepticus',


    # Parkinson's & movement disorders
    'NervousSystem_Parkinsons_Other_Movement', 

    # Multiple sclerosis & demyelinating
    'NervousSystem_Multiple_Sclerosis_Other_Demyelinating',  

    # Sleep & circadian (distributed, subtle)
    'NervousSystem_Sleep_Disorders',


]



In [4]:

analyzer = BBTransformerAnalyzer(
    base_dir='/mnt/movement/users/jaizor/xtra/data/fmri/chrt',
    weights_dir='/mnt/movement/users/jaizor/xtra/ΞΞ/__/weights', 
    initial_weights='weights_NervousX.pth',  
    min_composite=0.65,
    max_trials_per_disorder=3
)

results = analyzer.run_ordered_pipeline(TASKS)


PHASE 1/12: ASD
  📥 Loading pretrained weights: weights_NervousX.pth
  Trial 1/3...
STEP 1: Loading Data for Target = 'ASD'
  fMRI: /mnt/movement/users/jaizor/xtra/data/fmri/abide/fmri_ASD.npz
  Phenotype: /mnt/movement/users/jaizor/xtra/data/fmri/abide/pheno_ASD.csv
Loaded phenotype: (585, 4)
Loaded fMRI: (585, 150, 414)
  Subjects: 585
  Timepoints: 150
  Brain regions: 414
  ROI labels: 414
All required columns present

STEP 2: Preparing Data Loaders
Cohort: 585 subjects (271 cases, 314 controls, 46.3% prevalence)
Splits → Train: 409, Val: 88, Test: 88

Dataset Meta
  target: ASD
  n_total: 585
  n_positive: 271
  prevalence: 0.4632478654384613
  feature_dim: 414
  n_train: 409
  n_val: 88
  n_test: 88

STEP 3: Initializing BBTransformer
Model created on cuda with 29,354,880 parameters

STEP 3.5: Loading Pretrained Weights
  From: /mnt/movement/users/jaizor/xtra/ΞΞ/__/weights/weights_NervousX.pth
Attempting SAFE load (on CPU first): /mnt/movement/users/jaizor/xtra/ΞΞ/__/weights/wei

Early stopping at epoch 125 (F1: 0.7525)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_ASD.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.8182
  Precision: 0.7778
  Recall:    0.8537
  F1 Score:  0.8140
  ROC-AUC:   0.8544

Confusion Matrix:
[[37 10]
 [ 6 35]]

STEP 6: Permutation Importance (CSV)


Calculating permutation importance (f1) for 414 brain regions...


Permuting regions: 100%|██████████| 414/414 [54:18<00:00,  7.87s/it]


...

Early stopping at epoch 131 (F1: 0.6667)
✓ Saved SAFE weights (weights_only=True compatible) to: weights/weights_NervousSystem_Sleep_Disorders.pth

STEP 5: Test Set Evaluation



Performance Metrics:
  Accuracy:  0.6928
  Precision: 0.6538
  Recall:    0.8193
  F1 Score:  0.7273
  ROC-AUC:   0.7317

Confusion Matrix:
[[47 36]
 [15 68]]

STEP 6: Permutation Importance (CSV)


Calculating permutation importance (f1) for 414 brain regions...


Permuting regions: 100%|██████████| 414/414 [1:28:05<00:00, 12.77s/it]

Saved top 5 features to: results/importance_NervousSystem_Sleep_Disorders.csv
Permutation importance saved.
Results saved to JSON: results/NervousSystem_Sleep_Disorders_results.json

TRAINING & EVALUATION COMPLETE
Target: NervousSystem_Sleep_Disorders
  ✅ VALID MODEL FOUND (Composite: 0.7273)
  🔁 Weights updated for next disorder: /mnt/movement/users/jaizor/xtra/ΞΞ/__/weights/weights_NervousSystem_Sleep_Disorders.pth
